In [ ]:
# CELL 1 — PROJECT OBJECTIVE
PROJECT = "Physician Drug Adoption Prediction"
print("=" * 70)
print(PROJECT)
print("=" * 70)
print("""
Predict which currently non-adopting physicians are most likely to
adopt the drug for the first time in Quarter 11.

Primary business metric: Lift@20%
Secondary metrics: ROC-AUC, PR-AUC, Precision, Recall, F1
""")

In [ ]:
# CELL 2 — IMPORT LIBRARIES
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.inspection import permutation_importance
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier

print("Libraries imported successfully.")

In [ ]:
# CELL 3 — PATH CONFIGURATION
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT1_PATH = DATA_DIR / "Input_data_file1.csv"
INPUT2_PATH = DATA_DIR / "Input_data_file2.csv"
TEST_PATH = DATA_DIR / "Test_data_file.csv"

print("Data directory:", DATA_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())
print("If filenames differ, change the three paths above.")

In [ ]:
# CELL 4 — LOAD DATA
file1 = pd.read_csv(INPUT1_PATH)
file2 = pd.read_csv(INPUT2_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Input 1 shape:", file1.shape)
print("Input 2 shape:", file2.shape)
print("Test shape   :", test_df.shape)

In [ ]:
# CELL 5 — COLUMN DEFINITIONS
TARGET_COL = "brand_prescribed"
ID_COL = "physician_id"
QTR_COL = "year_quarter"

LAG_NUMERIC = [
    "total_representative_visits",
    "total_sample_dropped",
    "saving_cards_dropped",
    "vouchers_dropped",
    "total_seminar_as_attendee",
    "total_seminar_as_speaker",
    "total_prescriptions_for_indication1",
    "total_prescriptions_for_indication2",
    "total_prescriptions_for_indication3",
    "total_patient_with_commercial_insurance_plan",
    "total_patient_with_medicare_insurance_plan",
    "total_patient_with_medicaid_insurance_plan",
    "brand_web_impressions",
    "brand_ehr_impressions",
    "brand_enews_impressions",
    "brand_mobile_impressions",
    "brand_organic_web_visits",
    "brand_paidsearch_visits",
    "total_competitor_prescription",
    "new_prescriptions"
]

CATEGORICAL_COLS = [
    "physician_hospital_affiliation",
    "physician_in_group_practice",
    "physician_gender",
    "physician_speciality",
    "physician_value_tier"
]

STATIC_NUM_COLS = [
    "urban",
    "percent",
    "physician_age",
    "physician_years_experience"
]

In [ ]:
# CELL 6 — BASIC VALIDATION
required_input1 = set([ID_COL, QTR_COL, TARGET_COL] + LAG_NUMERIC)
required_input2 = set([ID_COL] + CATEGORICAL_COLS + STATIC_NUM_COLS)

missing_input1 = required_input1 - set(file1.columns)
missing_input2 = required_input2 - set(file2.columns)

assert not missing_input1, f"Missing Input 1 columns: {missing_input1}"
assert not missing_input2, f"Missing Input 2 columns: {missing_input2}"
print("Column validation passed.")

In [ ]:
# CELL 7 — INPUT 2 UNIQUENESS
assert file2[ID_COL].duplicated().sum() == 0, \
    "Input 2 must contain one profile row per physician."
print("Input 2 uniqueness check passed.")

In [ ]:
# CELL 8 — MERGE INPUT 1 + INPUT 2
rows_before = len(file1)

merged = file1.merge(
    file2,
    on=ID_COL,
    how="left",
    validate="m:1"
)

assert len(merged) == rows_before
assert merged.duplicated([ID_COL, QTR_COL]).sum() == 0

print("Merged shape:", merged.shape)
print("Merge validation passed.")

In [ ]:
# CELL 9 — PANEL COMPLETENESS
quarters_per_physician = merged.groupby(ID_COL)[QTR_COL].nunique()
print(quarters_per_physician.value_counts().sort_index())
assert (quarters_per_physician == 10).all()
print("Panel completeness passed.")

In [ ]:
# CELL 10 — MISSING VALUE EDA
missing_df = pd.DataFrame({
    "missing_count": merged.isna().sum(),
    "missing_pct": (merged.isna().mean() * 100).round(2)
}).sort_values("missing_pct", ascending=False)

display(missing_df)
print("Columns >=20% missing:")
display(missing_df[missing_df["missing_pct"] >= 20])

In [ ]:
# CELL 11 — ADOPTION TREND
adoption_trend = merged.groupby(QTR_COL)[TARGET_COL].agg(["mean", "count"])
adoption_trend.columns = ["adoption_rate", "n_physicians"]
display(adoption_trend.round(3))

plt.figure(figsize=(10, 5))
plt.plot(adoption_trend.index.astype(str), adoption_trend["adoption_rate"], marker="o")
plt.xticks(rotation=45)
plt.ylabel("Adoption Rate")
plt.xlabel("Quarter")
plt.title("Drug Adoption Rate by Quarter")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "adoption_trend.png", bbox_inches="tight")
plt.show()

In [ ]:
# CELL 12 — CLASS DISTRIBUTION
print(f"Overall adoption rate: {merged[TARGET_COL].mean():.2%}")
print(merged[TARGET_COL].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=merged, x=TARGET_COL)
plt.title("Current-Quarter Adoption Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# CELL 13 — NUMERICAL EDA
key_numeric = [
    "total_representative_visits",
    "total_sample_dropped",
    "new_prescriptions",
    "total_competitor_prescription"
]
display(merged[key_numeric].describe().round(2))
display(merged.groupby(TARGET_COL)[key_numeric].median().round(2))

In [ ]:
# CELL 14 — CATEGORICAL EDA
for col in CATEGORICAL_COLS:
    print(f"\n--- {col} ---")
    print(merged[col].value_counts(dropna=False).head(20))

In [ ]:
# CELL 15 — CHRONOLOGICAL TIME INDEX
merged[QTR_COL] = merged[QTR_COL].astype(str).str.zfill(6)
merged["year"] = merged[QTR_COL].str[:4].astype(int)
merged["quarter_number"] = merged[QTR_COL].str[4:6].astype(int)

# Quarterly chronological index
merged["time_index"] = (
    merged["year"] * 4 + merged["quarter_number"]
)

merged = merged.sort_values([ID_COL, "time_index"]).reset_index(drop=True)

unique_times = sorted(merged["time_index"].unique())
assert np.all(np.diff(unique_times) == 1)
print("Unique chronological indexes:", unique_times)

In [ ]:
# CELL 16 — NEXT-QUARTER TARGET
next_time = merged.groupby(ID_COL)["time_index"].shift(-1)
next_target = merged.groupby(ID_COL)[TARGET_COL].shift(-1)

merged["valid_next_qtr"] = (
    next_time == merged["time_index"] + 1
).astype(int)

merged["target_next_qtr"] = np.where(
    merged["valid_next_qtr"] == 1,
    next_target,
    np.nan
)

print("Valid next-quarter rows:", merged["valid_next_qtr"].sum())
print("Next-quarter adopters:", (merged["target_next_qtr"] == 1).sum())
print("Next-quarter non-adopters:", (merged["target_next_qtr"] == 0).sum())

In [ ]:
# CELL 17 — CURRENT NON-ADOPTERS / FIRST ADOPTION
previous_adoption = (
    merged.groupby(ID_COL)[TARGET_COL]
    .cummax()
    .shift(1, fill_value=0)
)

merged["adopted_before"] = previous_adoption
merged["adopted_first_qtr"] = (
    (merged[TARGET_COL] == 1) & (previous_adoption == 0)
)

merged["is_non_adopter_now"] = (
    (previous_adoption == 0) &
    (~merged["adopted_first_qtr"])
).astype(int)

model_data = merged[merged["is_non_adopter_now"] == 1].copy()
assert model_data["is_non_adopter_now"].eq(1).all()

print("Modeling rows:", len(model_data))
print("Physicians:", model_data[ID_COL].nunique())

In [ ]:
# CELL 18 — TARGET VALIDATION
target_rows = model_data[model_data["target_next_qtr"].notna()].copy()

next_actual = merged.set_index([ID_COL, "time_index"])[TARGET_COL]
mismatches = 0

for _, row in target_rows.iterrows():
    actual = next_actual.get(
        (row[ID_COL], int(row["time_index"]) + 1), np.nan
    )
    if pd.notna(actual) and int(actual) != int(row["target_next_qtr"]):
        mismatches += 1

print("Target mismatches:", mismatches)
assert mismatches == 0
print("Target construction verified.")

In [ ]:
# CELL 19 — LAG 1 / LAG 2
for col in LAG_NUMERIC:
    model_data[f"{col}_lag1"] = model_data.groupby(ID_COL)[col].shift(1)
    model_data[f"{col}_lag2"] = model_data.groupby(ID_COL)[col].shift(2)

lag1_cols = [f"{c}_lag1" for c in LAG_NUMERIC]
lag2_cols = [f"{c}_lag2" for c in LAG_NUMERIC]

assert len(lag1_cols) == 20
assert len(lag2_cols) == 20
print("Lag features created.")

In [ ]:
# CELL 20 — DERIVED FEATURES
for col in LAG_NUMERIC:
    l1, l2 = f"{col}_lag1", f"{col}_lag2"
    model_data[f"{col}_avg_2q"] = (model_data[l1] + model_data[l2]) / 2
    model_data[f"{col}_change"] = model_data[l1] - model_data[l2]

avg_cols = [f"{c}_avg_2q" for c in LAG_NUMERIC]
change_cols = [f"{c}_change" for c in LAG_NUMERIC]

print("Average and change features created.")
print("Sum features are excluded because sum = 2 × average.")

In [ ]:
# CELL 21 — LAG MISSINGNESS
lag_feature_cols = lag1_cols + lag2_cols + avg_cols + change_cols
display(
    model_data[lag_feature_cols]
    .isna().sum()
    .sort_values(ascending=False)
    .head(10)
)

print("Missing lag cells are handled by the model pipeline, not filled with zero.")

In [ ]:
# CELL 22 — FINAL FEATURE GROUPS
NUMERIC_FEATURES = (
    lag1_cols + lag2_cols + avg_cols + change_cols + STATIC_NUM_COLS
)
CAT_FEATURES = [c for c in CATEGORICAL_COLS if c in model_data.columns]
FEATURES = NUMERIC_FEATURES + CAT_FEATURES

assert TARGET_COL not in FEATURES
assert "target_next_qtr" not in FEATURES
assert ID_COL not in FEATURES
assert QTR_COL not in FEATURES

print("Total model features:", len(FEATURES))

In [ ]:
# CELL 23 — LABELED HISTORICAL DATA
labeled_data = model_data[model_data["target_next_qtr"].notna()].copy()
labeled_data["target_next_qtr"] = labeled_data["target_next_qtr"].astype(int)

LABELED_QTRS = sorted(labeled_data["time_index"].unique())

print("Labeled source quarters:", LABELED_QTRS)
assert len(LABELED_QTRS) == 9

In [ ]:
# CELL 24 — CHRONOLOGICAL 80/20 SPLIT
n_labeled_qtrs = len(LABELED_QTRS)
n_test_qtrs = max(1, int(np.ceil(n_labeled_qtrs * 0.20)))

split_index = n_labeled_qtrs - n_test_qtrs
DEV_QTRS = LABELED_QTRS[:split_index]
TEST_QTRS = LABELED_QTRS[split_index:]

assert set(DEV_QTRS).isdisjoint(TEST_QTRS)
assert max(DEV_QTRS) < min(TEST_QTRS)

print("Development quarters:", DEV_QTRS)
print("Historical test quarters:", TEST_QTRS)

In [ ]:
# CELL 25 — DEVELOPMENT / TEST DATASETS
dev = labeled_data[labeled_data["time_index"].isin(DEV_QTRS)].copy()
historical_test = labeled_data[
    labeled_data["time_index"].isin(TEST_QTRS)
].copy()

print("Development rows:", len(dev))
print("Historical test rows:", len(historical_test))
print("Development adoption rate:", f"{dev['target_next_qtr'].mean():.2%}")
print("Test adoption rate:", f"{historical_test['target_next_qtr'].mean():.2%}")

In [ ]:
# CELL 26 — CORRELATION ANALYSIS
corr_features = [c for c in NUMERIC_FEATURES if c in dev.columns]
dev_corr = dev[corr_features + ["target_next_qtr"]].corr(method="spearman")

target_corr = (
    dev_corr["target_next_qtr"]
    .drop("target_next_qtr")
    .sort_values(key=np.abs, ascending=False)
)

display(target_corr.head(20).to_frame("spearman_corr"))

In [ ]:
# CELL 27 — VIF DIAGNOSTIC
vif_candidates = [c for c in NUMERIC_FEATURES if c in dev.columns]

vif_sample = dev[vif_candidates].sample(
    n=min(10000, len(dev)), random_state=42
)

vif_sample = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(vif_sample),
    columns=vif_candidates
)

vif_sample = pd.DataFrame(
    StandardScaler().fit_transform(vif_sample),
    columns=vif_candidates
)

vif_results = []
for i, col in enumerate(vif_sample.columns):
    vif_results.append({
        "feature": col,
        "VIF": variance_inflation_factor(vif_sample.values, i)
    })

vif_df = pd.DataFrame(vif_results).sort_values("VIF", ascending=False)
display(vif_df.head(30))
vif_df.to_csv(OUTPUT_DIR / "development_vif.csv", index=False)

print("VIF is diagnostic; features are not automatically dropped solely due to high VIF.")

In [ ]:
# CELL 28 — TEMPORAL CV FOLDS
def create_temporal_folds(df, time_col="time_index"):
    quarters = sorted(df[time_col].unique())
    return [
        (quarters[:i], [quarters[i]])
        for i in range(2, len(quarters))
    ]

folds = create_temporal_folds(dev)

for i, (train_q, val_q) in enumerate(folds, 1):
    print(f"Fold {i}: Train={train_q}, Validation={val_q}")

In [ ]:
# CELL 29 — PREPROCESSING
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CAT_FEATURES)
])

print("Numeric: median imputation -> scaling")
print("Categorical: most-frequent imputation -> one-hot encoding")

In [ ]:
# CELL 30 — MODELS
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000, class_weight="balanced", C=1.0, solver="lbfgs"
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=12, class_weight="balanced",
        random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", random_state=42, n_jobs=-1
    )
}
print(list(models))

In [ ]:
# CELL 31 — METRICS
def lift_at_20(y_true, y_probability):
    y_true = np.asarray(y_true)
    y_probability = np.asarray(y_probability)
    n_top = int(np.ceil(len(y_true) * 0.20))
    ranking = np.argsort(y_probability)[::-1]
    top_indices = ranking[:n_top]

    overall_rate = y_true.mean()
    top_rate = y_true[top_indices].mean()

    return np.nan if overall_rate == 0 else top_rate / overall_rate


def evaluate_predictions(y_true, probabilities, threshold=0.50):
    predictions = (probabilities >= threshold).astype(int)

    return {
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "PR-AUC": average_precision_score(y_true, probabilities),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "Lift@20%": lift_at_20(y_true, probabilities)
    }

In [ ]:
# CELL 32 — TEMPORAL CROSS-VALIDATION
cv_results = []

for model_name, base_model in models.items():
    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    for fold_number, (train_quarters, validation_quarters) in enumerate(folds, 1):

        train_data = dev[dev["time_index"].isin(train_quarters)]
        validation_data = dev[dev["time_index"].isin(validation_quarters)]

        X_train = train_data[FEATURES]
        y_train = train_data["target_next_qtr"].astype(int)

        X_valid = validation_data[FEATURES]
        y_valid = validation_data["target_next_qtr"].astype(int)

        # New pipeline every fold prevents preprocessing leakage.
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", base_model)
        ])

        pipeline.fit(X_train, y_train)
        probabilities = pipeline.predict_proba(X_valid)[:, 1]

        metrics = evaluate_predictions(y_valid, probabilities)
        metrics.update({
            "model": model_name,
            "fold": fold_number,
            "train_start": min(train_quarters),
            "train_end": max(train_quarters),
            "validation_quarter": validation_quarters[0]
        })

        cv_results.append(metrics)

cv_results_df = pd.DataFrame(cv_results)
display(cv_results_df.round(4))

cv_results_df.to_csv(
    OUTPUT_DIR / "temporal_cv_results.csv", index=False
)

In [ ]:
# CELL 33 — MODEL COMPARISON / SELECTION
summary = (
    cv_results_df.groupby("model")
    .agg(
        mean_ROC_AUC=("ROC-AUC", "mean"),
        mean_PR_AUC=("PR-AUC", "mean"),
        mean_Precision=("Precision", "mean"),
        mean_Recall=("Recall", "mean"),
        mean_F1=("F1", "mean"),
        mean_Lift20=("Lift@20%", "mean"),
        std_Lift20=("Lift@20%", "std")
    )
    .reset_index()
    .sort_values("mean_Lift20", ascending=False)
)

display(summary.round(4))

BEST_MODEL_NAME = summary.iloc[0]["model"]

print("Selected model:", BEST_MODEL_NAME)
print("Primary selection criterion: mean Lift@20%")

In [ ]:
# CELL 34 — FIT BEST MODEL ON DEVELOPMENT DATA
best_model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models[BEST_MODEL_NAME])
])

X_dev = dev[FEATURES]
y_dev = dev["target_next_qtr"].astype(int)

best_model_pipeline.fit(X_dev, y_dev)
print("Development model fitted.")

In [ ]:
# CELL 35 — FINAL HISTORICAL TEST
X_test = historical_test[FEATURES]
y_test = historical_test["target_next_qtr"].astype(int)

test_probability = best_model_pipeline.predict_proba(X_test)[:, 1]
final_test_metrics = evaluate_predictions(y_test, test_probability)

print("=" * 60)
print("FINAL HISTORICAL TEST")
print("=" * 60)
for metric, value in final_test_metrics.items():
    print(f"{metric:15s}: {value:.4f}")

In [ ]:
# CELL 36 — CONFUSION MATRIX
test_prediction = (test_probability >= 0.50).astype(int)

cm = confusion_matrix(y_test, test_prediction)

display(pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
))

print("Threshold 0.50 is used here; business targeting is based on ranking/top 20%.")

In [ ]:
# CELL 37 — FEATURE IMPORTANCE ON DEVELOPMENT DATA
feature_names = (
    best_model_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

fitted_model = best_model_pipeline.named_steps["model"]

if hasattr(fitted_model, "feature_importances_"):
    native_importance = fitted_model.feature_importances_
elif hasattr(fitted_model, "coef_"):
    native_importance = np.abs(fitted_model.coef_[0])
else:
    native_importance = np.zeros(len(feature_names))

perm = permutation_importance(
    best_model_pipeline,
    X_dev,
    y_dev,
    scoring="average_precision",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "native_importance": native_importance,
    "permutation_importance": perm.importances_mean
}).sort_values("permutation_importance", ascending=False)

display(feature_importance_df.head(20).round(5))

feature_importance_df.to_csv(
    OUTPUT_DIR / "feature_importance_development.csv", index=False
)

In [ ]:
# CELL 38 — RETRAIN FINAL MODEL ON ALL LABELED HISTORY
all_labeled_history = labeled_data.copy()

final_model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models[BEST_MODEL_NAME])
])

X_all = all_labeled_history[FEATURES]
y_all = all_labeled_history["target_next_qtr"].astype(int)

final_model_pipeline.fit(X_all, y_all)

print("Final model trained on all labeled history Q1->Q2 through Q9->Q10.")
print("Q10->Q11 is excluded from supervised training because Q11 outcome is unknown.")

In [ ]:
# CELL 39 — Q11 PHYSICIAN POPULATION
q11_physicians = test_df[[ID_COL]].drop_duplicates()

print("Q11 physicians:", len(q11_physicians))
assert len(q11_physicians) == 1502

In [ ]:
# CELL 40 — VERIFY Q11 POPULATION
q11_history = merged[
    merged[ID_COL].isin(q11_physicians[ID_COL])
]

historical_adoption = q11_history.groupby(ID_COL)[TARGET_COL].max()
previous_adopters = historical_adoption[historical_adoption == 1]

print("Q11 physicians with previous adoption:", len(previous_adopters))
assert len(previous_adopters) == 0

In [ ]:
# CELL 41 — BUILD Q11 LAG-1 / LAG-2
q11_panel = merged[
    merged[ID_COL].isin(q11_physicians[ID_COL])
].copy()

Q10_INDEX = max(merged["time_index"].unique())
Q9_INDEX = Q10_INDEX - 1

q10_data = q11_panel[q11_panel["time_index"] == Q10_INDEX].copy()
q9_data = q11_panel[q11_panel["time_index"] == Q9_INDEX].copy()

assert len(q10_data) == 1502
assert len(q9_data) == 1502

q11_features = q11_physicians.copy()

q10_selected = q10_data[[ID_COL] + LAG_NUMERIC].rename(
    columns={c: f"{c}_lag1" for c in LAG_NUMERIC}
)

q9_selected = q9_data[[ID_COL] + LAG_NUMERIC].rename(
    columns={c: f"{c}_lag2" for c in LAG_NUMERIC}
)

q11_features = q11_features.merge(
    q10_selected, on=ID_COL, how="left", validate="1:1"
)

q11_features = q11_features.merge(
    q9_selected, on=ID_COL, how="left", validate="1:1"
)

print("Q11 Lag-1 = Q10")
print("Q11 Lag-2 = Q9")

In [ ]:
# CELL 42 — Q11 LAG VALIDATION
for col in LAG_NUMERIC:
    expected = q10_data[[ID_COL, col]].rename(columns={col: "expected"})
    check = q11_features[[ID_COL, f"{col}_lag1"]].merge(expected, on=ID_COL)
    assert np.allclose(check[f"{col}_lag1"], check["expected"], equal_nan=True)

for col in LAG_NUMERIC:
    expected = q9_data[[ID_COL, col]].rename(columns={col: "expected"})
    check = q11_features[[ID_COL, f"{col}_lag2"]].merge(expected, on=ID_COL)
    assert np.allclose(check[f"{col}_lag2"], check["expected"], equal_nan=True)

print("Q11 lag validation passed.")

In [ ]:
# CELL 43 — Q11 DERIVED FEATURES
for col in LAG_NUMERIC:
    l1, l2 = f"{col}_lag1", f"{col}_lag2"
    q11_features[f"{col}_avg_2q"] = (
        q11_features[l1] + q11_features[l2]
    ) / 2
    q11_features[f"{col}_change"] = (
        q11_features[l1] - q11_features[l2]
    )

print("Q11 derived features created.")

In [ ]:
# CELL 44 — Q11 STATIC FEATURES
static_profile = file2[
    [ID_COL] + STATIC_NUM_COLS + CAT_FEATURES
].drop_duplicates(ID_COL)

q11_features = q11_features.merge(
    static_profile,
    on=ID_COL,
    how="left",
    validate="1:1"
)

assert len(q11_features) == 1502
print("Static physician features added.")

In [ ]:
# CELL 45 — Q11 FEATURE COMPATIBILITY
missing_features = set(FEATURES) - set(q11_features.columns)
print("Missing features:", missing_features)

assert len(missing_features) == 0
print("Q11 feature compatibility passed.")

In [ ]:
# CELL 46 — Q11 PREDICTION
q11_probability = final_model_pipeline.predict_proba(
    q11_features[FEATURES]
)[:, 1]

predictions = pd.DataFrame({
    ID_COL: q11_features[ID_COL],
    "predicted_adoption_probability": q11_probability
})

assert len(predictions) == 1502
print("Q11 physicians scored:", len(predictions))

In [ ]:
# CELL 47 — RANK AND PRIORITIZE
predictions = predictions.sort_values(
    "predicted_adoption_probability",
    ascending=False
).reset_index(drop=True)

predictions["rank"] = np.arange(len(predictions)) + 1

n_physicians = len(predictions)
high_cutoff = int(np.ceil(0.20 * n_physicians))
medium_cutoff = int(np.ceil(0.50 * n_physicians))

predictions["target_group"] = np.select(
    [
        predictions["rank"] <= high_cutoff,
        predictions["rank"] <= medium_cutoff
    ],
    [
        "High Priority",
        "Medium Priority"
    ],
    default="Low Priority"
)

display(predictions.head(20))
print("High Priority:", high_cutoff)
print("Medium Priority:", medium_cutoff - high_cutoff)
print("Low Priority:", n_physicians - medium_cutoff)

In [ ]:
# CELL 48 — EXPORT OUTPUTS
prediction_output = OUTPUT_DIR / "physician_q11_adoption_predictions.csv"
predictions.to_csv(prediction_output, index=False)

pd.DataFrame([final_test_metrics]).to_csv(
    OUTPUT_DIR / "final_historical_test_metrics.csv",
    index=False
)

summary.to_csv(
    OUTPUT_DIR / "model_comparison.csv",
    index=False
)

print("Prediction output:", prediction_output.resolve())

In [ ]:
# CELL 49 — FINAL VALIDATION CHECKLIST
checks = {
    "No duplicate physician-quarter rows":
        merged.duplicated([ID_COL, QTR_COL]).sum() == 0,

    "Labeled rows have targets":
        labeled_data["target_next_qtr"].notna().all(),

    "Only current non-adopters modeled":
        model_data["is_non_adopter_now"].eq(1).all(),

    "Development/test quarters do not overlap":
        set(DEV_QTRS).isdisjoint(TEST_QTRS),

    "Development is earlier than test":
        max(DEV_QTRS) < min(TEST_QTRS),

    "Exactly 1,502 Q11 physicians":
        len(q11_features) == 1502,

    "Target excluded from features":
        TARGET_COL not in FEATURES and "target_next_qtr" not in FEATURES,

    "Physician ID excluded from features":
        ID_COL not in FEATURES,

    "No Lag-3/Lag-4 features":
        all("_lag3" not in c and "_lag4" not in c for c in FEATURES)
}

check_df = pd.DataFrame({
    "check": list(checks.keys()),
    "passed": list(checks.values())
})

display(check_df)
assert check_df["passed"].all()

print("ALL FINAL CHECKS PASSED.")

In [ ]:
# CELL 50 — FINAL SUMMARY
print("=" * 75)
print("FINAL PROJECT SUMMARY")
print("=" * 75)
print(f"""
Model: {BEST_MODEL_NAME}

Historical labeled windows:
Q1->Q2 through Q9->Q10

Chronological development/test:
Development = earlier ~80% of labeled quarters
Test = latest ~20% of labeled quarters

Feature engineering:
Lag-1, Lag-2, 2-quarter average, Lag1-Lag2 change,
static physician characteristics.

Preprocessing:
Numeric -> median imputation -> scaling
Categorical -> most-frequent imputation -> one-hot encoding

Model selection:
Primary = Lift@20%

Final Q11 prediction:
Lag-1 = Q10
Lag-2 = Q9
Physicians scored = 1,502

Targeting:
Top 20% = High Priority
Next 30% = Medium Priority
Bottom 50% = Low Priority
""")